In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier


data = pd.read_csv('Graph metrics.csv') 


X = data.drop(['Class', 'SUBID'], axis=1)
y = data['Class']

FileNotFoundError: [Errno 2] No such file or directory: 'Graph metrics.csv'

In [2]:
feature_mapping = {feature: idx for idx, feature in enumerate(X.columns)}

X.columns = [feature_mapping[feature] for feature in X.columns]

feature_mapping_df = pd.DataFrame(list(feature_mapping.items()), columns=['Feature Name', 'Numerical Value'])
feature_mapping_df.to_excel('feature_mapping.xlsx', index=False)

X.to_csv('modified_X.csv', index=False)

,0,1,2,3,4,5,6,7,8,9,...,390,391,392,393,394,395,396,397,398,399
0,0.142792,0.152380,0.138242,0.128318,0.073354,0.125638,0.117033,0.124513,0.085654,0.077673,...,28.113633,23.492334,20.554582,25.886703,26.186172,25.914882,25.870608,28.302477,26.197549,26.891909
1,0.136982,0.087800,0.136468,0.100557,0.083036,0.092927,0.115858,0.157892,0.104102,0.164333,...,31.421620,31.447779,20.616437,28.188898,28.470461,26.680460,31.738922,19.939570,30.836562,27.995663
2,0.087111,0.085090,0.090624,0.149344,0.081675,0.119827,0.196860,0.181025,0.110351,0.071053,...,28.310651,20.361887,33.036005,28.187911,30.290679,24.875032,25.918969,31.745262,27.781989,30.162745
3,0.069815,0.096431,0.125917,0.062097,0.058270,0.116637,0.068922,0.119892,0.053945,0.059635,...,31.890400,28.726099,19.449664,23.991417,24.264801,29.453199,23.937690,28.405901,23.939032,23.772156
4,0.092352,0.099120,0.110039,0.194233,0.115064,0.099066,0.100000,0.094404,0.087217,0.140740,...,29.338520,25.306874,20.678923,28.353024,28.433054,28.156925,29.280365,19.847805,28.399285,26.308604
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278,0.092889,0.095617,0.139340,0.150831,0.097366,0.169972,0.120007,0.122656,0.103204,0.091839,...,26.251046,26.551413,23.091714,31.684692,30.099486,31.318400,31.067078,30.298989,31.113116,31.101114
279,0.081681,0.103816,0.076791,0.106884,0.101864,0.110500,0.101984,0.135339,0.099337,0.089232,...,25.150330,28.763525,30.448833,29.622317,29.153205,30.090540,29.753290,17.360859,29.257596,28.106458
280,0.088771,0.127531,0.058040,0.135893,0.101317,0.063221,0.077533,0.092528,0.077417,0.058553,...,26.517966,28.219569,28.678872,26.227687,25.617300,26.808445,27.312468,25.163550,26.750995,25.708057
281,0.108748,0.102547,0.069388,0.066918,0.175202,0.109014,0.089036,0.093743,0.066168,0.064809,...,23.260392,23.107140,28.988568,25.237874,24.755406,27.370672,28.465443,29.037736,28.388063,27.818975


In [3]:
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X) 
X=X_normalized

array([[0.1681835 , 0.16320496, 0.17932762, ..., 0.75440794, 0.30300964,
        0.41195951],
       [0.15845044, 0.06336355, 0.17598886, ..., 0.27664682, 0.49339708,
        0.45693343],
       [0.07491981, 0.05917381, 0.08975396, ..., 0.95108944, 0.36803584,
        0.54523411],
       ...,
       [0.07769965, 0.12478746, 0.02846239, ..., 0.57508546, 0.32572334,
        0.36372185],
       [0.11116129, 0.08616221, 0.04980954, ..., 0.79641224, 0.39290943,
        0.44973405],
       [0.02938081, 0.02399757, 0.06134854, ..., 0.69147393, 0.35622205,
        0.44992717]])

In [4]:
features_arr = [5, 10, 20, 40, 50, 75, 100, 125, 150, 175, 200, 225, 250, 275, 300, 325, 350, 375, 400, 425, 450]

param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 4, 5],
    'reg_lambda': [5, 10, 15], 
    'reg_alpha': [0, 1, 5],   
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]  
}

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


train_results = []
test_results = []

for num_features in features_arr:
    print(f"Testing with top {num_features} features")
    
    # Outer loop
    all_test_metrics = []
    all_train_metrics = []
    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Inner cross-validation for feature selection using Random Forest
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rfecv = RFECV(estimator=rf, step=5, cv=5, n_jobs=-1)
        rfecv.fit(X_train, y_train)
        
        # Select top 'num_features' based on importance
        ranking = np.argsort(rfecv.ranking_)[:num_features]
        X_train_selected = X_train[:, ranking]
        X_test_selected = X_test[:, ranking]
        selected_features = list(ranking)
        selected_features_str = ", ".join(map(str, selected_features))

        # Perform GridSearchCV
        grid_search = GridSearchCV(
            XGBClassifier(random_state=42, eval_metric='logloss'),
            param_grid, cv=5, n_jobs=-1, scoring='roc_auc'
        )
        grid_search.fit(X_train_selected, y_train)
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_

        # Predict on test set
        y_test_pred = best_model.predict(X_test_selected)
        y_test_pred_proba = best_model.predict_proba(X_test_selected)[:, 1]
        auc_test = roc_auc_score(y_test, y_test_pred_proba)

        # Calculate test metrics
        tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
        accuracy_test = accuracy_score(y_test, y_test_pred)
        precision_test = precision_score(y_test, y_test_pred, average='weighted')
        recall_test = recall_score(y_test, y_test_pred, average='weighted')
        f1_test = f1_score(y_test, y_test_pred, average='weighted')
        specificity_test = tn / (tn + fp)
        
        # Predict on train set
        y_train_pred = best_model.predict(X_train_selected)
        y_train_pred_proba = best_model.predict_proba(X_train_selected)[:, 1]
        auc_train = roc_auc_score(y_train, y_train_pred_proba)

        # Calculate train metrics
        tn, fp, fn, tp = confusion_matrix(y_train, y_train_pred).ravel()
        accuracy_train = accuracy_score(y_train, y_train_pred)
        precision_train = precision_score(y_train, y_train_pred, average='weighted')
        recall_train = recall_score(y_train, y_train_pred, average='weighted')
        f1_train = f1_score(y_train, y_train_pred, average='weighted')
        specificity_train = tn / (tn + fp)

       
        test_results.append([
            num_features, selected_features_str, fold + 1,
            accuracy_test, precision_test, recall_test, 
            f1_test, auc_test, specificity_test, best_params
        ])

       
        train_results.append([
            num_features, selected_features_str, fold + 1,
            accuracy_train, precision_train, recall_train, 
            f1_train, auc_train, specificity_train, best_params
        ])

        
        all_test_metrics.append([
            accuracy_test, precision_test, recall_test, 
            f1_test, auc_test, specificity_test
        ])

        all_train_metrics.append([
            accuracy_train, precision_train, recall_train, 
            f1_train, auc_train, specificity_train
        ])

   
    avg_test_metrics = np.mean(all_test_metrics, axis=0)
    avg_train_metrics = np.mean(all_train_metrics, axis=0)

    
    test_results.append([
        num_features, selected_features_str, 'Average',
        *avg_test_metrics, best_params
    ])

    train_results.append([
        num_features, selected_features_str, 'Average',
        *avg_train_metrics, best_params
    ])


columns = [
    'Top Features', 'Selected Features', 'Fold', 
    'Accuracy', 'Precision', 'Recall', 'F1 Score', 
    'AUC', 'Specificity', 'Best Parameters'
]

train_df = pd.DataFrame(train_results, columns=columns)
test_df = pd.DataFrame(test_results, columns=columns)


train_df.to_excel('train_results_XGBoost.xlsx', index=False)
test_df.to_excel('test_results_XGBoost.xlsx', index=False)

Testing with top 5 features
Testing with top 10 features
Testing with top 20 features
Testing with top 40 features
Testing with top 50 features
Testing with top 75 features
Testing with top 100 features
Testing with top 125 features
Testing with top 150 features
Testing with top 175 features
Testing with top 200 features
Testing with top 225 features
Testing with top 250 features
Testing with top 275 features
Testing with top 300 features
Testing with top 325 features
Testing with top 350 features
Testing with top 375 features
Testing with top 400 features
✅ Train results saved to 'train_results_XGB_RFECV.xlsx'
✅ Test results saved to 'test_results_XGB_RFECV.xlsx'
Fold 5/5 - Features 400 - Best AUC: 0.846
